# Drone Swarm Agentic Command Module

A multi-agent simulation framework for autonomous drone swarm mission planning, execution, and post-mission analysis.

**Architecture overview:**

| Agent | Role |
|---|---|
| `MissionPlanner` | Clusters raw target coordinates into optimised waypoints via K-Means |
| `TargetIdentifier` | Processes imagery and returns detected objects above a confidence threshold |
| `ThreatAssessor` | Fuses multi-sensor readings into a single 0–1 threat score |
| `NavigationAgent` | Computes obstacle-avoiding paths with a pure-NumPy A\* implementation |
| `SwarmCoordinator` | Assigns the closest available drone to each mission target |
| `PostMissionAnalyzer` | Aggregates outcomes and generates a structured mission report |
| `CommandModule` | Orchestrates the full pipeline; exposes safety-override controls |

**What changed from the original notebook:**
- Fixed `asyncio` event-loop crash (`asyncio.run()` inside a running loop)
- Fixed `AttributeError: 'Grid' object has no attribute 'matrix'` — replaced `pathfinding` library with a self-contained A\* implementation (zero extra dependencies)
- Fixed inverted obstacle matrix (original had `>500 = 1 = traversable`; corrected to `>500 = obstacle`)
- Fixed `sensor_data` variable scoping bug (variable used in `report.update` was undefined in one code path)
- Replaced bare `Dict` sensor data with a typed `SensorData` dataclass
- Replaced string-literal status values with `DroneStatus` / `MissionOutcome` enums (eliminates silent typo bugs)
- Added `__post_init__` validation on `DroneState` and `SensorData`
- Replaced `print()` throughout with `logging` (structured, levelled, per-class loggers)
- Made drone initialisation deterministic and reproducible via seeded RNG
- `SwarmCoordinator` now respects safety-mode flags before assignment
- `NavigationAgent.set_terrain_data` signature aligned with call sites
- All K-Means calls pass `n_init=10` to silence sklearn warning


In [ ]:
# Install required third-party packages (pre-installed on Colab; uncomment locally)
# !pip install numpy scikit-learn

# Verify versions
import numpy, sklearn
print(f"numpy {numpy.__version__}   scikit-learn {sklearn.__version__}")

In [ ]:
# Standard library
import asyncio
import heapq
import logging
import platform
import random
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Dict, List, Optional, Tuple

# Third-party  (available in Colab without extra installs)
import numpy as np
from sklearn.cluster import KMeans


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)-8s | %(name)-22s | %(message)s",
    force=True,   # re-apply in Colab where root logger may already have handlers
)


## Enumerations

In [ ]:
class DroneStatus(str, Enum):
    ACTIVE    = "active"
    INACTIVE  = "inactive"
    RETURNING = "returning"
    LOST      = "lost"


class MissionOutcome(str, Enum):
    SUCCESS            = "success"
    FAILED             = "failed"
    HUMAN_CONTROLLED   = "human_controlled"
    SELF_DESTRUCT      = "self_destruct_initiated"
    ABORTED            = "aborted"


## Data models

In [ ]:
@dataclass
class DroneState:
    """Represents the live state of a single drone."""
    id:       int
    position: Tuple[float, float, float]   # (x, y, altitude) in metres
    battery:  float                         # 0.0 – 1.0
    status:   DroneStatus
    payload:  Dict[str, float]
    human_override_active:          bool = False
    self_destruct_sequence_initiated: bool = False

    def __post_init__(self) -> None:
        if not (0.0 <= self.battery <= 1.0):
            raise ValueError(f"battery={self.battery!r} must be in [0, 1]")


@dataclass
class Mission:
    """A planned mission with target waypoints and operational constraints."""
    id:          str
    targets:     List[Tuple[float, float]]
    priority:    int
    constraints: Dict[str, float]
    created_at:  datetime = field(default_factory=datetime.now)


@dataclass
class SensorData:
    """Typed container for multi-modal sensor readings."""
    timestamp:                  datetime
    location:                   Tuple[float, float, float]
    radar_contact_count:        int
    radar_strength:             float          # 0.0 – 1.0
    infrared_signatures:        List[Dict]
    acoustic_loudness:          float          # 0.0 – 1.0
    electronic_warfare_signals: List[str]

    def __post_init__(self) -> None:
        self.radar_strength   = max(0.0, min(1.0, self.radar_strength))
        self.acoustic_loudness = max(0.0, min(1.0, self.acoustic_loudness))


## A\* pathfinder

Pure-NumPy implementation — no external `pathfinding` package required.
Supports 8-directional movement and automatically finds the nearest walkable
cell when start/end falls inside an obstacle zone.

In [ ]:
class AStarPathfinder:
    """2-D grid A* with 8-directional movement and obstacle avoidance.

    Parameters
    ----------
    obstacle_grid : np.ndarray  (H × W)
        Truthy cells are obstacles; falsy cells are traversable.
    resolution : float
        Metres per grid cell (used for world↔grid coordinate conversion).
    """

    def __init__(self, obstacle_grid: np.ndarray, resolution: float = 1.0) -> None:
        self.obstacle_grid = obstacle_grid.astype(bool)
        self.resolution    = resolution
        self.rows, self.cols = obstacle_grid.shape

    # ── coordinate helpers ────────────────────────────────────────────────────

    def _heuristic(self, a: Tuple[int, int], b: Tuple[int, int]) -> float:
        return np.hypot(b[0] - a[0], b[1] - a[1])

    def _world_to_grid(self, x: float, y: float) -> Tuple[int, int]:
        row = max(0, min(int(y / self.resolution), self.rows - 1))
        col = max(0, min(int(x / self.resolution), self.cols - 1))
        return row, col

    def _grid_to_world(self, row: int, col: int) -> Tuple[float, float]:
        return col * self.resolution, row * self.resolution

    # ── helpers ───────────────────────────────────────────────────────────────

    def _nearest_walkable(self, row: int, col: int) -> Tuple[int, int]:
        """BFS outward to find the closest traversable cell."""
        for radius in range(1, max(self.rows, self.cols)):
            for dr in range(-radius, radius + 1):
                for dc in range(-radius, radius + 1):
                    nr, nc = row + dr, col + dc
                    if 0 <= nr < self.rows and 0 <= nc < self.cols:
                        if not self.obstacle_grid[nr, nc]:
                            return nr, nc
        return row, col   # fallback: return original if grid is fully blocked

    def _reconstruct(
        self,
        came_from: Dict[Tuple, Tuple],
        current: Tuple[int, int],
    ) -> List[Tuple[float, float]]:
        path = [self._grid_to_world(*current)]
        while current in came_from:
            current = came_from[current]
            path.append(self._grid_to_world(*current))
        path.reverse()
        return path

    # ── public interface ──────────────────────────────────────────────────────

    def find_path(
        self,
        start: Tuple[float, float],
        end:   Tuple[float, float],
    ) -> List[Tuple[float, float]]:
        """Return a list of (x, y) world-coordinate waypoints from start to end.

        Falls back to a two-point direct line if no path exists.
        """
        sr, sc = self._world_to_grid(*start)
        er, ec = self._world_to_grid(*end)

        if self.obstacle_grid[sr, sc]:
            sr, sc = self._nearest_walkable(sr, sc)
        if self.obstacle_grid[er, ec]:
            er, ec = self._nearest_walkable(er, ec)

        open_heap: List            = []
        came_from: Dict            = {}
        g_score:   Dict            = {(sr, sc): 0.0}

        _counter = 0   # tie-breaker: avoids tuple comparison when f-scores are equal
        heapq.heappush(open_heap, (0.0, _counter, (sr, sc)))

        _DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1),
                 (-1, -1), (-1, 1), (1, -1), (1, 1)]

        while open_heap:
            _, __, current = heapq.heappop(open_heap)
            if current == (er, ec):
                return self._reconstruct(came_from, current)

            cr, cc = current
            for dr, dc in _DIRS:
                nr, nc = cr + dr, cc + dc
                if not (0 <= nr < self.rows and 0 <= nc < self.cols):
                    continue
                if self.obstacle_grid[nr, nc]:
                    continue
                step       = 1.414 if (dr != 0 and dc != 0) else 1.0
                tentative_g = g_score[current] + step
                neighbor   = (nr, nc)
                if tentative_g < g_score.get(neighbor, float("inf")):
                    came_from[neighbor] = current
                    g_score[neighbor]   = tentative_g
                    f = tentative_g + self._heuristic(neighbor, (er, ec))
                    _counter += 1
                    heapq.heappush(open_heap, (f, _counter, neighbor))

        # No path found — return a direct two-point line as a graceful fallback
        return [start, end]


## Agents

In [ ]:
class MissionPlanner:
    """Converts raw target coordinates into a clustered Mission object."""

    def __init__(self) -> None:
        self.missions: List[Mission] = []
        self._log = logging.getLogger(self.__class__.__name__)

    def plan_mission(self, objectives: Dict, terrain_data: np.ndarray) -> Mission:
        mission_id = f"M{len(self.missions) + 1}_{datetime.now().strftime('%Y%m%d%H%M%S')}"
        targets    = self._prioritize_targets(objectives, terrain_data)
        mission    = Mission(
            id=mission_id, targets=targets,
            priority=1, constraints={"max_altitude": 500.0},
        )
        self.missions.append(mission)
        self._log.info("Planned mission %s with %d targets", mission_id, len(targets))
        return mission

    def _prioritize_targets(
        self,
        objectives:   Dict,
        terrain_data: np.ndarray,
    ) -> List[Tuple[float, float]]:
        raw = objectives.get("targets", [])
        if not raw:
            return []
        coords    = np.array([[t["lat"], t["lon"]] for t in raw])
        n_clusters = min(5, len(coords))
        if n_clusters <= 1:
            return [tuple(coords[0])]
        km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(coords)
        return [tuple(c) for c in km.cluster_centers_]


# ─────────────────────────────────────────────────────────────────────────────

class TargetIdentifier:
    """Processes imagery and returns detected objects above a confidence threshold.

    The model backend is deliberately abstract: swap `_infer` for any real
    classifier (TensorFlow, PyTorch, ONNX) without touching the rest of the
    pipeline.
    """

    def __init__(self) -> None:
        self._log = logging.getLogger(self.__class__.__name__)

    def _infer(self, patch: np.ndarray) -> np.ndarray:
        """Mock inference.  Replace with actual model.predict() call."""
        seed = int(np.sum(patch[:5, :5]) * 1000) % (2 ** 31)
        return np.random.default_rng(seed).random(10)

    async def identify_targets(
        self,
        image:                np.ndarray,
        confidence_threshold: float = 0.7,
    ) -> List[Dict]:
        if image.ndim != 3 or image.shape[2] != 3:
            self._log.warning("Unexpected image shape %s — skipping", image.shape)
            return []

        h, w   = image.shape[:2]
        sh, sw = max(1, h // 224), max(1, w // 224)
        patch  = image[::sh, ::sw][:224, :224].astype(np.float32) / 255.0
        ph     = max(0, 224 - patch.shape[0])
        pw     = max(0, 224 - patch.shape[1])
        if ph > 0 or pw > 0:
            patch = np.pad(patch, ((0, ph), (0, pw), (0, 0)))

        scores = self._infer(patch)
        return [
            {
                "id":         i,
                "type":       "detected_object",
                "confidence": float(s),
                "position":   (float(i * 20), float(i * 15)),
            }
            for i, s in enumerate(scores)
            if s > confidence_threshold
        ]


# ─────────────────────────────────────────────────────────────────────────────

class ThreatAssessor:
    """Fuses multi-sensor readings into a single 0–1 threat score."""

    _WEIGHTS: Dict[str, float] = {
        "radar_contact_count":        0.20,
        "radar_strength":             0.30,
        "infrared_signatures":        0.25,
        "acoustic_loudness":          0.15,
        "electronic_warfare_signals": 0.10,
    }
    _HIGH_THREAT_SIGNALS = frozenset({"jamming", "tracking_radar"})

    def __init__(self) -> None:
        self._log = logging.getLogger(self.__class__.__name__)

    async def assess_threats(self, sensor_data: SensorData) -> float:
        t  = min(sensor_data.radar_contact_count / 10.0, 1.0) * self._WEIGHTS["radar_contact_count"]
        t += sensor_data.radar_strength                        * self._WEIGHTS["radar_strength"]
        t += min(len(sensor_data.infrared_signatures) / 5.0, 1.0) * self._WEIGHTS["infrared_signatures"]
        t += sensor_data.acoustic_loudness                     * self._WEIGHTS["acoustic_loudness"]
        if any(s in sensor_data.electronic_warfare_signals for s in self._HIGH_THREAT_SIGNALS):
            t += 0.20   # hard boost for active jamming / tracking radar
        t += min(len(sensor_data.electronic_warfare_signals) / 3.0, 1.0) * self._WEIGHTS["electronic_warfare_signals"]
        return float(min(max(t, 0.0), 1.0))


# ─────────────────────────────────────────────────────────────────────────────

class NavigationAgent:
    """Computes obstacle-aware paths using an A* planner on a terrain grid."""

    def __init__(self) -> None:
        self._pathfinder: Optional[AStarPathfinder] = None
        self._log = logging.getLogger(self.__class__.__name__)

    def set_terrain_data(
        self,
        terrain_data:       np.ndarray,
        resolution:         float = 10.0,
        obstacle_threshold: float = 500.0,
    ) -> None:
        """Build the traversability grid from an elevation map.

        Cells with elevation > obstacle_threshold are treated as obstacles.
        """
        obstacles         = (terrain_data > obstacle_threshold).astype(np.uint8)
        self._pathfinder  = AStarPathfinder(obstacles, resolution=resolution)
        self._log.info(
            "Terrain grid %s loaded — resolution=%.1f m/cell, obstacle_threshold=%.0f m",
            obstacles.shape, resolution, obstacle_threshold,
        )

    async def compute_path(
        self,
        start: Tuple[float, float],
        end:   Tuple[float, float],
    ) -> List[Tuple[float, float]]:
        if self._pathfinder is None:
            self._log.error("Terrain not initialised — returning direct path")
            return [start, end]
        path = self._pathfinder.find_path(start, end)
        self._log.debug("Path %s → %s: %d waypoints", start, end, len(path))
        return path


# ─────────────────────────────────────────────────────────────────────────────

class SwarmCoordinator:
    """Assigns the nearest available drone to each mission target."""

    def __init__(self) -> None:
        self._log = logging.getLogger(self.__class__.__name__)

    async def coordinate_swarm(
        self,
        mission: Mission,
        drones:  List[DroneState],
    ) -> List[Dict]:
        available = [
            d for d in drones
            if (
                d.status == DroneStatus.ACTIVE
                and d.battery > 0.2
                and not d.human_override_active
                and not d.self_destruct_sequence_initiated
            )
        ]
        assignments: List[Dict] = []
        for target in mission.targets:
            if not available:
                self._log.warning("No drones available for remaining targets")
                break
            best = min(
                available,
                key=lambda d: float(
                    np.linalg.norm(np.array(d.position[:2]) - np.array(target))
                ),
            )
            dist = float(np.linalg.norm(np.array(best.position[:2]) - np.array(target)))
            assignments.append({
                "drone_id":      best.id,
                "target":        target,
                "estimated_time": dist / 10.0,   # assumes 10 m/s cruise speed
            })
            available.remove(best)

        self._log.info(
            "%d assignments generated from %d active drones",
            len(assignments), len(drones),
        )
        return assignments


# ─────────────────────────────────────────────────────────────────────────────

class PostMissionAnalyzer:
    """Aggregates per-drone outcomes into a structured mission report."""

    def analyze(
        self,
        mission:      Mission,
        outcomes:     List[Dict],
        threat_level: float,
    ) -> Dict:
        if not outcomes:
            return {
                "mission_id":       mission.id,
                "success_rate":     0.0,
                "total_targets":    len(mission.targets),
                "completed_targets": 0,
                "failed_targets":   0,
                "threat_level":     threat_level,
                "timestamp":        datetime.now().isoformat(),
                "recommendations":  ["No outcomes recorded for this mission"],
            }

        successes = [o for o in outcomes if o.get("status") == MissionOutcome.SUCCESS]
        rate      = len(successes) / len(outcomes)

        recs: List[str] = []
        if rate < 0.5:
            recs.append("High failure rate — review mission planning parameters")
        elif rate < 0.7:
            recs.append("Sub-optimal performance — consider equipment or tactical upgrades")
        elif rate > 0.9:
            recs.append("Excellent performance — consider escalating mission complexity")
        if threat_level > 0.7:
            recs.append("High-threat environment — evaluate stealth routing or abort criteria")

        safety_modes = [
            o["drone_id"] for o in outcomes
            if o.get("status") in {MissionOutcome.HUMAN_CONTROLLED, MissionOutcome.SELF_DESTRUCT}
        ]
        if safety_modes:
            recs.append(f"Safety modes triggered for drones: {safety_modes}")

        return {
            "mission_id":            mission.id,
            "success_rate":          rate,
            "total_targets":         len(mission.targets),
            "completed_targets":     len(successes),
            "failed_targets":        len(outcomes) - len(successes),
            "threat_level":          threat_level,
            "safety_modes_triggered": safety_modes,
            "timestamp":             datetime.now().isoformat(),
            "recommendations":       recs,
        }


## Command module

In [ ]:
class CommandModule:
    """Central orchestrator for the drone swarm mission pipeline.

    Owns all sub-agents and exposes the full Plan → Identify → Assess →
    Coordinate → Navigate → Execute → Analyse workflow as a single async call.
    """

    def __init__(self, num_drones: int = 5) -> None:
        self._log         = logging.getLogger(self.__class__.__name__)
        self.planner      = MissionPlanner()
        self.identifier   = TargetIdentifier()
        self.navigator    = NavigationAgent()
        self.assessor     = ThreatAssessor()
        self.coordinator  = SwarmCoordinator()
        self.analyzer     = PostMissionAnalyzer()
        self.num_drones   = num_drones
        self._drones: List[DroneState] = []
        self._log.info("CommandModule initialised with %d drones", num_drones)

    # ── drone management ──────────────────────────────────────────────────────

    def _spawn_drones(self, seed: int = 42) -> List[DroneState]:
        """Create a fresh fleet with deterministic, reproducible state."""
        rng = random.Random(seed)
        return [
            DroneState(
                id       = i,
                position = (
                    rng.uniform(9.0, 20.0),    # x aligned to target latitude range
                    rng.uniform(19.0, 30.0),   # y aligned to target longitude range
                    rng.uniform(50, 200),       # altitude in metres
                ),
                battery  = rng.uniform(0.6, 1.0),
                status   = DroneStatus.ACTIVE,
                payload  = {"fuel": rng.uniform(0.8, 1.0)},
            )
            for i in range(self.num_drones)
        ]

    # ── safety controls ───────────────────────────────────────────────────────

    def trigger_human_override(self, drone_id: int) -> None:
        """Flag a drone for human takeover; it will be skipped in path planning."""
        for d in self._drones:
            if d.id == drone_id:
                d.human_override_active = True
                self._log.warning(
                    "Human override activated for drone %d", drone_id
                )
                return
        self._log.error("trigger_human_override: drone %d not found", drone_id)

    def initiate_self_destruct(self, drone_id: int) -> None:
        """Mark a drone for self-destruct; execution is recorded but not carried out."""
        for d in self._drones:
            if d.id == drone_id:
                d.self_destruct_sequence_initiated = True
                self._log.warning(
                    "Self-destruct sequence initiated for drone %d", drone_id
                )
                return
        self._log.error("initiate_self_destruct: drone %d not found", drone_id)

    # ── mission pipeline ──────────────────────────────────────────────────────

    async def execute_mission(
        self,
        objectives:   Dict,
        terrain_data: np.ndarray,
        image_data:   np.ndarray,
    ) -> Dict:
        """Run the full mission pipeline and return a structured report dict."""
        self._log.info("=== Mission execution started ===")

        # 1. Load terrain into the navigation agent
        self.navigator.set_terrain_data(terrain_data, resolution=10.0)

        # 2. Plan mission from objectives
        mission = self.planner.plan_mission(objectives, terrain_data)

        # 3. Identify targets from imagery
        identified = await self.identifier.identify_targets(image_data)
        self._log.info(
            "Target identification: %d objects above threshold", len(identified)
        )

        # 4. Assess threat level from multi-sensor data
        rng = random.Random()
        sensor = SensorData(
            timestamp             = datetime.now(),
            location              = (0.0, 0.0, 100.0),
            radar_contact_count   = rng.randint(0, 15),
            radar_strength        = rng.uniform(0.1, 1.0),
            infrared_signatures   = [
                {"strength": rng.uniform(0.2, 1.0)}
                for _ in range(rng.randint(0, 5))
            ],
            acoustic_loudness     = rng.uniform(0.1, 1.0),
            electronic_warfare_signals = rng.sample(
                ["none", "comm", "radar", "jamming", "tracking_radar"],
                k=rng.randint(0, 3),
            ),
        )
        threat = await self.assessor.assess_threats(sensor)
        self._log.info("Threat level assessed at %.2f", threat)

        # 5. Spawn drones only if fleet not pre-configured (preserves safety flags set before call)
        if not self._drones:
            self._drones = self._spawn_drones()
        assignments = await self.coordinator.coordinate_swarm(mission, self._drones)

        # 6. Compute navigation paths
        paths: Dict[int, List] = {}
        for a in assignments:
            drone         = next(d for d in self._drones if d.id == a["drone_id"])
            paths[drone.id] = await self.navigator.compute_path(
                drone.position[:2], a["target"]
            )
        self._log.info("Navigation: %d paths computed", len(paths))

        # 7. Simulate execution outcomes
        outcomes: List[Dict] = []
        for a in assignments:
            drone = next(d for d in self._drones if d.id == a["drone_id"])
            if drone.self_destruct_sequence_initiated:
                status = MissionOutcome.SELF_DESTRUCT
            elif drone.human_override_active:
                status = MissionOutcome.HUMAN_CONTROLLED
            else:
                p_success = max(0.05, (1.0 - threat) * drone.battery)  # 5% floor: always non-zero chance
                status    = (
                    MissionOutcome.SUCCESS
                    if random.random() < p_success
                    else MissionOutcome.FAILED
                )
            outcomes.append({
                "drone_id":       drone.id,
                "target":         a["target"],
                "status":         status,
                "execution_time": a["estimated_time"],
                "battery_used":   random.uniform(0.05, 0.25),
                "waypoints":      len(paths.get(drone.id, [])),
            })

        # 7b. Record outcomes for safety-mode drones excluded from assignment.
        #     Without this, their flags never appear in the mission report.
        assigned_ids = {a["drone_id"] for a in assignments}
        for drone in self._drones:
            if drone.id in assigned_ids:
                continue
            if drone.self_destruct_sequence_initiated:
                outcomes.append({"drone_id": drone.id, "target": None,
                                 "status": MissionOutcome.SELF_DESTRUCT,
                                 "execution_time": 0.0, "battery_used": 0.0, "waypoints": 0})
            elif drone.human_override_active:
                outcomes.append({"drone_id": drone.id, "target": None,
                                 "status": MissionOutcome.HUMAN_CONTROLLED,
                                 "execution_time": 0.0, "battery_used": 0.0, "waypoints": 0})

        # 8. Analyse and return report
        report = self.analyzer.analyze(mission, outcomes, threat)
        report["identified_targets"] = len(identified)
        report["drones_deployed"]    = len(assignments)
        self._log.info(
            "Mission complete — success rate: %.1f%%",
            report["success_rate"] * 100,
        )
        return report


## Run the mission

In [ ]:
async def main() -> Dict:
    cmd = CommandModule(num_drones=5)

    objectives = {
        "targets": [
            {"lat": 10.5, "lon": 20.3},
            {"lat": 15.2, "lon": 25.7},
            {"lat": 12.8, "lon": 22.1},
            {"lat": 18.4, "lon": 28.9},
        ]
    }

    # Reproducible synthetic inputs
    terrain = np.random.default_rng(0).random((100, 100)) * 1000   # elevation in metres
    image   = (np.random.default_rng(1).random((540, 960, 3)) * 255).astype(np.uint8)

    report = await cmd.execute_mission(objectives, terrain, image)

    # ── pretty-print report ───────────────────────────────────────────────────
    width = 60
    print("\n" + "=" * width)
    print("MISSION REPORT".center(width))
    print("=" * width)
    for key, value in report.items():
        if key == "recommendations":
            continue
        print(f"  {key:<28} {value}")
    print()
    print("  recommendations:")
    for rec in report["recommendations"]:
        print(f"    • {rec}")
    print("=" * width)
    return report


# Compatible with Colab (running event loop), Jupyter, and plain Python.
if platform.system() == "Emscripten":
    asyncio.ensure_future(main())
else:
    try:
        asyncio.get_running_loop()
        await main()           # already inside a running loop (Colab / Jupyter)
    except RuntimeError:
        asyncio.run(main())    # no loop running — plain Python script


## Safety mode demonstration

Trigger human override and self-destruct on specific drones mid-mission and
verify that the outcomes are recorded correctly in the report.

In [ ]:
async def demo_safety_modes() -> None:
    cmd = CommandModule(num_drones=5)

    objectives = {
        "targets": [
            {"lat": 10.5, "lon": 20.3},
            {"lat": 15.2, "lon": 25.7},
            {"lat": 12.8, "lon": 22.1},
            {"lat": 18.4, "lon": 28.9},
        ]
    }
    terrain = np.random.default_rng(7).random((100, 100)) * 1000
    image   = (np.random.default_rng(8).random((540, 960, 3)) * 255).astype(np.uint8)

    # Reset fleet then pre-spawn — ensures repeated calls don't carry stale flags.
    cmd._drones = []                        # clear any existing fleet
    cmd._drones = cmd._spawn_drones()       # fresh deterministic spawn
    cmd.trigger_human_override(drone_id=0)
    cmd.initiate_self_destruct(drone_id=1)

    report = await cmd.execute_mission(objectives, terrain, image)
    print(f"\nSafety modes triggered for drones: {report['safety_modes_triggered']}")
    print(f"Recommendations: {report['recommendations']}")


if platform.system() == "Emscripten":
    asyncio.ensure_future(demo_safety_modes())
else:
    try:
        asyncio.get_running_loop()
        await demo_safety_modes()
    except RuntimeError:
        asyncio.run(demo_safety_modes())


## Next steps

| Area | Suggested upgrade |
|---|---|
| **Target identification** | Replace `_infer()` in `TargetIdentifier` with a real object-detection model (e.g. TensorFlow MobileNetV2, YOLOv8 via `ultralytics`) |
| **Threat model** | Train a scikit-learn `RandomForestClassifier` or small Keras network on historical sensor data; load in `ThreatAssessor.__init__` |
| **3-D navigation** | Extend `AStarPathfinder` to a voxel grid and add altitude profiling per waypoint |
| **Assignment optimisation** | Replace greedy nearest-drone assignment with Hungarian algorithm (`scipy.optimize.linear_sum_assignment`) for global optimality |
| **Communication modelling** | Add a `CommModel` agent that can drop messages and test re-planning under comms loss |
| **Persistent state** | Model battery depletion, fuel consumption, and return-to-base triggers across multiple missions |
| **Simulation visualisation** | Plot terrain grid, drone positions, and computed paths with `matplotlib` after each mission |
| **Dynamic threats** | Simulate pop-up threats that appear during execution and trigger re-routing |
